# HistoQL — Historical Research Query Language

This notebook is our report for Part I of the project. It contains the specification of the language and its complete implementation, and it runs from top to bottom. It keeps two layers separate:

1. **HistoQL**, the readable frontend language used by historians.
2. **SQL**, a parameterised backend target generated only after parsing and semantic validation.

The SQL schema below is an explicit *demonstration adapter*, not part of the language definition. Changing a database should require changing only the adapter/compiler, never the HistoQL grammar.

Running it requires Lark and the LarkAG extension used in the course:

```
pip install lark "git+https://github.com/nmacedo/LarkAG.git@ag-soundness-fixes"
```

The only other files it needs are the four demonstration CSV files, kept in a `data` folder next to this notebook:

```
data/entities.csv
data/relationships.csv
data/trade_ww2_entities.csv
data/trade_ww2_relationships.csv
```

## 1. Language design

### Domain and purpose

HistoQL is a domain-specific language for historical research. Its purpose is to let a researcher state a historical question directly, in terms of people, events, places and sources, without knowing how the data is stored. A question such as "who took part in the Liberal Revolution, in Porto, between 1820 and 1834?" would otherwise require SQL, and therefore knowledge of tables, foreign keys and joins.

The intended users are historians and historical researchers. The tasks we target are finding, filtering and relating historical information, and chaining research steps according to what earlier steps returned.

HistoQL is not a general-purpose language: it has no arithmetic, no functions and no general assignment. `IF` and `FOR EACH` exist only to chain queries over research results.

### Main concepts

| Concept | Representation in the language |
|---|---|
| Entity kinds | `people`, `groups`, `events`, `places`, `institutions`, `sources` |
| Relationships | Predicates after `WHO`/`THAT`, such as `participated_in` or `mentions` |
| Place | `IN "Porto"` |
| Period | `DURING 1820..1834` |
| Attributes of an entity | `WHERE birth_year >= 1800` and `RETURN name, role` |
| Research results | Named with `LET`, then tested with `IF` or traversed with `FOR EACH` |

### Example program

```histoql
RESEARCH "Portuguese Liberalism" {
  LET politicians = FIND people
    WHO participated_in "Liberal Revolution"
    IN "Porto"
    DURING 1820..1834
    RETURN name, role;

  IF EXISTS politicians {
    FIND sources THAT mentions "Liberal Revolution" RETURN title, date;
  }
}
```

`LET`, `IF`, and `FOR EACH` meet the course requirements for declarations, scope, branching, and repetition. The query clauses (`WHO`/`THAT`, `IN`, `DURING`, `WHERE`, `RETURN`) express the historical domain.

### Further representative programs

A conditional with an `ELSE` branch and a count:

```histoql
RESEARCH "Liberal events in Porto" {
  LET liberal_events = FIND events IN "Porto" DURING 1820..1834 RETURN name, date;

  IF liberal_events COUNT >= 2 {
    FIND sources THAT mentions "Liberal Revolution" RETURN title, author;
  } ELSE {
    FIND events DURING 1800..1850 RETURN name;
  }
}
```

Repetition over a stored result:

```histoql
LET politicians = FIND people WHO participated_in "Liberal Revolution" RETURN name, role;

FOR EACH politician IN politicians {
  FIND institutions IN "Lisboa" RETURN name, type;
}
```

Several filters in one query, in any order:

```histoql
FIND people WHERE birth_year >= 1770 WHO lived_in "Porto" DURING 1800..1834 RETURN name, occupation;
```

### How the language meets the minimum requirements

| Requirement | In HistoQL |
|---|---|
| Variables and declarations | `LET name = query;` |
| Scopes and a scoping discipline | Blocks delimited by `{ }`. A block sees outer names, and what it declares does not leave it. |
| Expressions and state | Comparisons in `WHERE`, counts in `COUNT` and periods in `DURING`. The state is the set of stored results. |
| Compound and nested constructs | `IF` and `FOR EACH` blocks can contain any statement, inside a `RESEARCH` block |
| Branching and repetition | `IF ... ELSE` and `FOR EACH` |
| Properties not determined by syntax | Inverted periods, invalid fields and relations for an entity, repeated or undeclared names (section 3) |

### Design decisions

**Entity kinds are terminals; relations and fields are identifiers.** The six entity kinds are a closed vocabulary, so we fixed them in the syntax, and `FIND animals;` is a syntax error. Relations and fields depend on the entity being queried. A context-free grammar could only express this by multiplying rules per entity, so we check them with context conditions, and the lists can grow without adding productions.

**`LET` binds a name to the entity kind of its result.** This is the information the semantic analysis propagates through the program.

**Scopes are block-based and without shadowing.** A declaration cannot reuse a name inherited from an enclosing block, so each name has a single meaning at any point of a program.

**The language is separate from its target.** SQL is one possible backend, generated by a replaceable adapter. Changing the database does not change the grammar.

In [1]:
from __future__ import annotations

import csv
import json
import os
import sqlite3
from typing import Any

from lark import Lark, Transformer
from lark.exceptions import UnexpectedInput
from lark_ag import Lark_AG, ContextConditionError


# We use two error classes so that a problem of form and a problem of meaning stay distinct.
class HistoQLSyntaxError(Exception):
    """The input does not conform to the context-free grammar."""


class HistoQLSemanticError(Exception):
    """The parsed program violates a HistoQL context condition."""

## 2. Lexical and syntactic specification

### Lexical structure

| Token class | Definition | Examples |
|---|---|---|
| Keywords | Fixed uppercase words | `RESEARCH`, `LET`, `FIND`, `WHO`, `THAT`, `IN`, `DURING`, `WHERE`, `RETURN`, `IF`, `ELSE`, `EXISTS`, `COUNT`, `FOR`, `EACH` |
| Entity kinds | Fixed lowercase words, with priority 10 | `people`, `sources` |
| `ID` | `/[a-zA-Z_][a-zA-Z0-9_]*/` | `politicians`, `birth_year` |
| `STRING` | Quoted text with escapes | `"Porto"` |
| `INT` | Integer | `1820` |
| `COMPOP` | Comparison operators | `=`, `!=`, `<`, `<=`, `>`, `>=` |
| Punctuation | Fixed symbols | `{`, `}`, `;`, `,`, `=`, `..` |
| Ignored | Whitespace and `#` comments | `# a comment` |

An entity word such as `people` also matches the `ID` pattern. We resolve this by giving the entity terminals a higher priority (`.10`), so the lexer classifies them as entity kinds.

### Context-free grammar

The language has a single grammar, in the cell below. Its productions are in pure BNF: repetition is written with recursion, and alternatives are separate productions, with no `*`, `+` or `?` operators.

```
stmts: stmts statement
     |
```

Besides being the form we were asked to use, this has a concrete advantage: each step of a list is a tree node, which gives attributes a place to flow from one statement to the next. Recursion is always on the left, which suits LALR parsing because the parser reduces after each element.

The same grammar is used for the specification and the implementation. A program that violates it is rejected by this grammar, and no later stage repeats its checks.

## 3. Semantic specification

### Properties that syntax cannot determine

`FIND people DURING 1834..1820;` respects the production `"DURING" year ".." year`, yet a period cannot end before it starts. A context-free grammar has no memory of what surrounds a production, so it cannot compare the two years, know which entity a clause applies to, or remember which names were declared.

We specify these properties with an attribute grammar, following the model of notebook NB06: attributes, evaluation rules (ER) and context conditions (CC). We do not use translation rules; SQL generation is a separate stage.

### Where each error is detected

Every error is visible in the grammar. A **syntax error** means that no production matches the input. A **semantic error** means that a production matched, but one of its context conditions failed.

| Kind | Error | Production | Check | Rejected program |
|---|---|---|---|---|
| Syntax | Missing `;` | `querystmt: query ";"` | No production matches | `FIND people RETURN name` |
| Syntax | Unknown entity kind | `entity` | No production matches | `FIND animals;` |
| Syntax | Clause after `RETURN` | `query` | No production matches | `FIND people RETURN name IN "Porto";` |
| Semantic | Inverted period | `duringclause` | `year[1].value <= year[2].value` | `FIND people DURING 1834..1820;` |
| Semantic | Field not in the entity | `fieldref` | `str(ID) in {...}[fieldref.ent]` | `FIND people RETURN cor_dos_olhos;` |
| Semantic | Relation not valid for the entity | `relationclause` | `str(ID) in {...}[relationclause.ent]` | `FIND places WHO ruled "x";` |
| Semantic | Repeated or shadowed name | `letstmt` | `str(ID) not in letstmt.env_in` | `LET a = FIND people; LET a = FIND places;` |
| Semantic | Undeclared name in `IF` | `condition` (2 alternatives) | `str(ID) in condition.env_in` | `IF EXISTS nope { FIND people; }` |
| Semantic | Undeclared collection | `foreachstmt` | `collection.name in foreachstmt.env_in` | `FOR EACH x IN nope { FIND people; }` |
| Semantic | Loop variable is not new | `foreachstmt` | `loopvar.name not in foreachstmt.env_in` | `LET a = FIND people; FOR EACH a IN a { FIND events; }` |
| Semantic | No default field | `query` (without `RETURN`) | `entity.kind != "sources"` | `FIND sources;` |

### Attributes

| Attribute | Kind | Symbols | Meaning |
|---|---|---|---|
| `env_in` | Inherited | `stmts`, `statement`, `letstmt`, `ifstmt`, `foreachstmt`, `condition` | Names known before the construct, each with its entity kind |
| `env_out` | Synthesized | `stmts`, `statement`, `letstmt` | Names known after the construct |
| `ent` | Inherited | `clauses`, `relationclause`, `whereclause`, `returnclause`, `returnfields`, `fieldref` | Entity being queried |
| `kind` | Synthesized | `entity`, `query` | Entity returned by the query |
| `value` | Synthesized | `year` | Numeric value of a year |
| `n` | Synthesized | `clauses` | Number of filter clauses |
| `name` | Synthesized | `loopvar`, `collection` | The two names in a `FOR EACH` |

In total there are 21 attribute occurrences (12 inherited, 9 synthesized), 50 evaluation rules and 9 context conditions.

In [2]:
# The HistoQL grammar: pure BNF productions, with attributes, evaluation rules (ER)
# and context conditions (CC), in Lark_AG notation.
HISTOQL_GRAMMAR = r'''
# Attribute declarations. SA = synthesized (flows up the tree), IA = inherited (flows down the tree).
SA(year) { value : int }
SA(loopvar) { name : str }
SA(collection) { name : str }
SA(entity) { kind : str }
SA(query) { kind : str }

SA(clauses) { n : int }
IA(clauses) { ent : str }
IA(relationclause) { ent : str }
IA(whereclause) { ent : str }
IA(returnclause) { ent : str }
IA(returnfields) { ent : str }
IA(fieldref) { ent : str }

SA(letstmt) { env_out : dict[str, str] }
IA(letstmt) { env_in : dict[str, str] }

SA(statement) { env_out : dict[str, str] }
IA(statement) { env_in : dict[str, str] }

SA(stmts) { env_out : dict[str, str] }
IA(stmts) { env_in : dict[str, str] }

IA(ifstmt) { env_in : dict[str, str] }
IA(foreachstmt) { env_in : dict[str, str] }
IA(condition) { env_in : dict[str, str] }

# A program starts with an empty environment.
start: researchblock
     | stmts
ER {
	stmts.env_in = {};
}

researchblock: "RESEARCH" STRING "{" stmts "}"
ER {
	stmts.env_in = {};
}

# The environment entering a statement is the one leaving the previous statement.
stmts: stmts statement
ER {
	stmts[2].env_in = stmts[1].env_in;
	statement.env_in = stmts[2].env_out;
	stmts[1].env_out = statement.env_out;
}
|
ER {
	stmts.env_out = stmts.env_in;
}

# Only LET changes the environment. For IF and FOR EACH, env_out = env_in: what a block declares does not leave it.
statement: letstmt
ER {
	letstmt.env_in = statement.env_in;
	statement.env_out = letstmt.env_out;
}
| querystmt
ER {
	statement.env_out = statement.env_in;
}
| ifstmt
ER {
	ifstmt.env_in = statement.env_in;
	statement.env_out = statement.env_in;
}
| foreachstmt
ER {
	foreachstmt.env_in = statement.env_in;
	statement.env_out = statement.env_in;
}

# SEMANTIC ERROR: a name cannot be declared twice, or reuse a name inherited from an outer block.
letstmt: "LET" ID "=" query ";"
ER {
	letstmt.env_out = letstmt.env_in | { str(ID): query.kind };
}
CC {
	str(ID) not in letstmt.env_in -> "Duplicate declaration in this scope";
}

querystmt: query ";"

ifstmt: "IF" condition "{" stmts "}"
ER {
	condition.env_in = ifstmt.env_in;
	stmts.env_in = ifstmt.env_in;
}
| "IF" condition "{" stmts "}" "ELSE" "{" stmts "}"
ER {
	condition.env_in = ifstmt.env_in;
	stmts[1].env_in = ifstmt.env_in;
	stmts[2].env_in = ifstmt.env_in;
}

# SEMANTIC ERRORS: the collection must have been declared, and the loop variable must be a new name.
# The loop variable exists only in the body and takes the entity kind of the collection.
foreachstmt: "FOR" "EACH" loopvar "IN" collection "{" stmts "}"
ER {
	stmts.env_in = foreachstmt.env_in | { loopvar.name: foreachstmt.env_in.get(collection.name, "") };
}
CC {
	collection.name in foreachstmt.env_in -> "Undeclared research result";
	loopvar.name not in foreachstmt.env_in -> "Duplicate declaration in this scope";
}

loopvar: ID
ER {
	loopvar.name = str(ID);
}

collection: ID
ER {
	collection.name = str(ID);
}

# The entity kind goes up to the query and down to its clauses.
# SEMANTIC ERROR: without RETURN the default field is name, which sources do not have.
query: "FIND" entity clauses
ER {
	query.kind = entity.kind;
	clauses.ent = entity.kind;
}
CC {
	entity.kind != "sources" -> "Sources have no default field: RETURN is required";
}
| "FIND" entity clauses returnclause
ER {
	query.kind = entity.kind;
	clauses.ent = entity.kind;
	returnclause.ent = entity.kind;
}

entity: PEOPLE
ER {
	entity.kind = "people";
}
| GROUPS
ER {
	entity.kind = "groups";
}
| EVENTS
ER {
	entity.kind = "events";
}
| PLACES
ER {
	entity.kind = "places";
}
| INSTITUTIONS
ER {
	entity.kind = "institutions";
}
| SOURCES
ER {
	entity.kind = "sources";
}

clauses: clauses relationclause
ER {
	clauses[2].ent = clauses[1].ent;
	relationclause.ent = clauses[1].ent;
	clauses[1].n = clauses[2].n + 1;
}
| clauses placeclause
ER {
	clauses[2].ent = clauses[1].ent;
	clauses[1].n = clauses[2].n + 1;
}
| clauses duringclause
ER {
	clauses[2].ent = clauses[1].ent;
	clauses[1].n = clauses[2].n + 1;
}
| clauses whereclause
ER {
	clauses[2].ent = clauses[1].ent;
	whereclause.ent = clauses[1].ent;
	clauses[1].n = clauses[2].n + 1;
}
|
ER {
	clauses.n = 0;
}

# SEMANTIC ERROR: each entity kind has its own legal relations.
relationclause: linkword ID value
CC {
	str(ID) in {"people": ["participated_in", "lived_in", "worked_at", "ruled", "member_of"],
    "groups": ["participated_in", "based_in", "allied_with", "ruled"],
    "events": ["occurred_in", "caused", "followed", "before", "after"],
    "places": ["part_of"],
    "institutions": ["located_in", "governed"],
    "sources": ["mentions", "produced_in"]}
    [relationclause.ent] -> "Invalid relation for this entity";
}

linkword: "WHO"
        | "THAT"

whereclause: "WHERE" fieldref COMPOP value
ER {
	fieldref.ent = whereclause.ent;
}

placeclause: "IN" STRING

# SEMANTIC ERROR: a period cannot end before it starts.
duringclause: "DURING" year ".." year
CC {
	year[1].value <= year[2].value -> "Invalid period";
}

returnclause: "RETURN" returnfields
ER {
	returnfields.ent = returnclause.ent;
}

returnfields: returnfields "," fieldref
ER {
	returnfields[2].ent = returnfields[1].ent;
	fieldref.ent = returnfields[1].ent;
}
| fieldref
ER {
	fieldref.ent = returnfields.ent;
}

# SEMANTIC ERROR: each entity kind has its own legal fields. WHERE and RETURN both use this rule.
fieldref: ID
CC {
	str(ID) in {"people": ["name", "role", "occupation", "birth_year", "death_year"],
    "groups": ["name", "type", "start_year", "end_year"],
    "events": ["name", "date", "type", "start_year", "end_year"],
    "places": ["name", "type"],
    "institutions": ["name", "type", "start_year", "end_year"],
    "sources": ["title", "date", "type", "author"]}
    [fieldref.ent] -> "Unknown field for this entity";
}

# SEMANTIC ERROR: a name tested by IF must have been declared.
condition: "EXISTS" ID
CC {
	str(ID) in condition.env_in -> "Undeclared research result";
}
| ID "COUNT" COMPOP INT
CC {
	str(ID) in condition.env_in -> "Undeclared research result";
}

value: STRING
     | INT

year: INT
ER {
	year.value = int(INT);
}

# Lexical rules. Priority 10 makes entity words win over the ID pattern.
PEOPLE.10: "people"
GROUPS.10: "groups"
EVENTS.10: "events"
PLACES.10: "places"
INSTITUTIONS.10: "institutions"
SOURCES.10: "sources"

COMPOP: "=" | "!=" | "<" | "<=" | ">" | ">="
ID: /[a-zA-Z_][a-zA-Z0-9_]*/
%import common.ESCAPED_STRING -> STRING
%import common.INT
%import common.WS
%ignore WS
%ignore /#[^\n]*/
'''

parser = Lark_AG(HISTOQL_GRAMMAR)

# Lark_AG uses Earley by default. We parse with LALR, which the grammar allows because it has no conflicts.
lalr_parser = Lark(parser.getCFG(), parser="lalr", start="start")
parser.processor.parser = lalr_parser
print("HistoQL grammar loaded.")

HistoQL grammar loaded.


### Dependencies and evaluation

The central dependency is in the list of statements, where the environment entering a statement is the environment leaving the previous one:

```
stmts: stmts statement
ER {
    stmts[2].env_in = stmts[1].env_in;
    statement.env_in = stmts[2].env_out;
    stmts[1].env_out = statement.env_out;
}
```

For `LET a = FIND people; LET a = FIND places;`, evaluation proceeds as follows:

1. `start` sets the environment to `{}`, and it descends to the empty list, where `env_out = env_in`.
2. The first `letstmt` receives `env_in = {}`.
3. `entity.kind = "people"` is synthesized into `query.kind`.
4. The condition `"a" not in {}` holds, and the rule produces `env_out = {"a": "people"}`.
5. The second statement inherits `env_in = {"a": "people"}`.
6. The condition `"a" not in {"a": "people"}` fails, and the program is rejected.

The scoping discipline follows from one equation. In the `ifstmt` and `foreachstmt` alternatives of `statement`, `statement.env_out = statement.env_in`: the block inherits the environment, but what it declares does not leave it. A `FOR EACH` also adds its loop variable to the environment of its body, with the entity kind of the collection, so the variable exists only inside the loop. We introduced the non-terminals `loopvar` and `collection` to tell the two identifiers of this production apart, and `fieldref` so that `WHERE` and `RETURN` share one list of fields.

### Well-definedness

Our attribute grammar is L-attributed. Every inherited attribute depends only on inherited attributes of the parent or on attributes of left siblings: for example, `statement.env_in = stmts[2].env_out`, `clauses.ent = entity.kind`, and the body of a `FOR EACH`, which uses `loopvar.name` and `collection.name`. It therefore has no circular dependencies, and its attributes can be evaluated in a single left-to-right, depth-first traversal.

### Coverage

The assignment asks for selected semantic properties. We cover declarations, scopes, entity-dependent vocabulary and periods. The type of a value in `WHERE` is not checked (section 5).

### Errors in practice

The next cell runs the grammar on accepted and rejected programs, and shows the kind of each error.

In [3]:
# Each program below is checked by the grammar above. "ok" means it was accepted.
def check(source):
    try:
        parser.process(source)
        print(f"ok              {source}")
    except UnexpectedInput as error:
        print(f"Syntax error    {source}\n                {str(error).splitlines()[0]}")
    except ContextConditionError as error:
        print(f"Semantic error  {source}\n                {error}")


print("--- Accepted ---")
check('FIND people WHO ruled "x" DURING 1800..1900;')
check('FIND people IN "Rome";')
check('LET a = FIND people; FOR EACH p IN a { IF EXISTS p { FIND events; } }')
check('LET a = FIND people; IF EXISTS a { LET b = FIND events; } LET b = FIND places;')

print("\n--- Syntax errors: no production matches ---")
check('FIND people RETURN name')
check('FIND animals;')
check('FIND people RETURN name IN "Porto";')

print("\n--- Semantic errors: a context condition fails ---")
check('FIND people DURING 1834..1820;')
check('FIND people RETURN cor_dos_olhos;')
check('FIND people WHERE age > 3;')
check('FIND places WHO ruled "x";')
check('LET a = FIND people; LET a = FIND places;')
check('LET a = FIND people; IF EXISTS a { LET a = FIND events; }')
check('IF EXISTS nope { FIND people; }')
check('IF nope COUNT > 1 { FIND people; }')
check('FOR EACH x IN nope { FIND people; }')
check('LET a = FIND people; FOR EACH a IN a { FIND events; }')
check('FIND sources;')

--- Accepted ---
ok              FIND people WHO ruled "x" DURING 1800..1900;
ok              FIND people IN "Rome";
ok              LET a = FIND people; FOR EACH p IN a { IF EXISTS p { FIND events; } }
ok              LET a = FIND people; IF EXISTS a { LET b = FIND events; } LET b = FIND places;

--- Syntax errors: no production matches ---
Syntax error    FIND people RETURN name
                Unexpected token Token('$END', '') at line 1, column 20.
Syntax error    FIND animals;
                Unexpected token Token('ID', 'animals') at line 1, column 6.
Syntax error    FIND people RETURN name IN "Porto";
                Unexpected token Token('IN', 'IN') at line 1, column 25.

--- Semantic errors: a context condition fails ---
Semantic error  FIND people DURING 1834..1820;
                Invalid period
Semantic error  FIND people RETURN cor_dos_olhos;
                Unknown field for this entity
Semantic error  FIND people WHERE age > 3;
                Unknown field for this ent

## 4. Implementation

Part I does not require an implementation, but we built one to test the language. It uses the grammar of section 2 and adds only what comes after validation:

| Stage | Input | Output | Code |
|---|---|---|---|
| Parsing and semantic validation | HistoQL text | Annotated parse tree, or an error | `HISTOQL_GRAMMAR`, `parser` |
| AST construction | Parse tree | Nested dictionaries | `ToAST`, `parse_histoql` |
| SQL generation | Query in the AST | SQL text and parameters | `compile_query_to_sql` |
| Execution | SQL on the demonstration data | Result tables | `create_archive_database`, `execute_program` |

`histoql()` at the end of the section runs a whole program through these stages.

### 4.1 Grammar tests

These tests only ask whether the productions accept the text, so they use the LALR parser directly, and names need not be declared here.

In [4]:
def test_valid(label, code):
    """The input should be accepted by the grammar."""
    try:
        lalr_parser.parse(code)
        print(f"[PASS] {label}")
        return True
    except UnexpectedInput as error:
        print(f"[FAIL] {label}: {error.__class__.__name__}")
        return False


def test_invalid(label, code):
    """The input should be rejected by the grammar."""
    try:
        lalr_parser.parse(code)
        print(f"[FAIL] {label}: expected rejection")
        return False
    except UnexpectedInput:
        print(f"[PASS] {label}: rejected as expected")
        return True


valid_tests = [
    ("Basic query",
     'FIND people RETURN name;'),

    ("Query without RETURN",
     'FIND events;'),

    ("Multiple return attributes",
     'FIND people RETURN name, birth_year;'),

    ("Temporal filter",
     'FIND people DURING 1800..1850 RETURN name;'),

    ("Location filter",
     'FIND people IN "Portugal" RETURN name;'),

    ("WHO relation clause",
     'FIND people WHO occupation "scientist" RETURN name;'),

    ("THAT relation clause",
     'FIND people THAT occupation "scientist" RETURN name;'),

    ("WHERE comparison",
     'FIND people WHERE birth_year >= 1800 RETURN name;'),

    ("Multiple clauses",
     'FIND people IN "France" DURING 1800..1900 '
     'WHERE birth_year >= 1850 RETURN name;'),

    ("LET statement",
     'LET researchers = FIND people RETURN name;'),

    ("IF with EXISTS",
     'IF EXISTS researchers { FIND people RETURN name; }'),

    ("IF with COUNT",
     'IF people COUNT >= 2 { FIND people RETURN name; }'),

    ("IF with ELSE",
     'IF EXISTS researchers { FIND people; } '
     'ELSE { FIND events; }'),

    ("FOR EACH loop",
     'FOR EACH person IN researchers { FIND people RETURN name; }'),

    ("RESEARCH block",
     'RESEARCH "Historical study" { '
     'FIND people DURING 1800..1900 RETURN name; }'),

    ("Comment before query",
     '# Test query\nFIND people RETURN name;'),
]

invalid_tests = [
    ("Missing semicolon",
     'FIND people RETURN name'),

    ("Unknown entity",
     'FIND animals RETURN name;'),

    ("Missing entity",
     'FIND RETURN name;'),

    ("Missing attribute after RETURN",
     'FIND people RETURN;'),

    ("Missing WHERE value",
     'FIND people WHERE birth_year >= RETURN name;'),

    ("Missing second year",
     'FIND people DURING 1834.. RETURN name;'),

    ("Missing relation value",
     'FIND people WHO occupation RETURN name;'),

    ("Missing location value",
     'FIND people IN RETURN name;'),

    ("IF without a block",
     'IF EXISTS researchers FIND people;'),

    ("Incomplete FOR EACH loop",
     'FOR EACH person IN { FIND people; }'),

    ("Unclosed RESEARCH block",
     'RESEARCH "Study" { FIND people;'),

    ("Invalid comparison operator",
     'FIND people WHERE birth_year === 1800;'),
]


def run_unit_tests():
    print("=== VALID INPUT TESTS ===")
    results = [test_valid(label, code) for label, code in valid_tests]
    print("\n=== INVALID INPUT TESTS ===")
    results += [test_invalid(label, code) for label, code in invalid_tests]
    print(f"\n{sum(results)} of {len(results)} tests passed")
    return results


run_unit_tests();

=== VALID INPUT TESTS ===
[PASS] Basic query
[PASS] Query without RETURN
[PASS] Multiple return attributes
[PASS] Temporal filter
[PASS] Location filter
[PASS] WHO relation clause
[PASS] THAT relation clause
[PASS] WHERE comparison
[PASS] Multiple clauses
[PASS] LET statement
[PASS] IF with EXISTS
[PASS] IF with COUNT
[PASS] IF with ELSE
[PASS] FOR EACH loop
[PASS] RESEARCH block
[PASS] Comment before query

=== INVALID INPUT TESTS ===
[PASS] Missing semicolon: rejected as expected
[PASS] Unknown entity: rejected as expected
[PASS] Missing entity: rejected as expected
[PASS] Missing attribute after RETURN: rejected as expected
[PASS] Missing WHERE value: rejected as expected
[PASS] Missing second year: rejected as expected
[PASS] Missing relation value: rejected as expected
[PASS] Missing location value: rejected as expected
[PASS] IF without a block: rejected as expected
[PASS] Incomplete FOR EACH loop: rejected as expected
[PASS] Unclosed RESEARCH block: rejected as expected
[PASS] I

### 4.2 From the parse tree to the AST

The parse tree reflects every production, including the recursion of the lists. `ToAST` turns it into nested dictionaries with only what later stages need. It runs after the grammar has accepted the program.

In [5]:
class ToAST(Transformer):
    # Lark_AG names each alternative of a rule rule_0, rule_1, and so on, and Lark calls the method
    # with that name, from the leaves up. Keywords and punctuation are already gone from items.

    # Tokens: convert the text of each token into a Python value.
    def STRING(self, token):
        return json.loads(str(token))  # removes the quotes and resolves escapes

    def INT(self, token):
        return int(token)

    def ID(self, token):
        return str(token)

    def COMPOP(self, token):
        return str(token)

    # Rules that only wrap one value.
    def year_0(self, items):
        return items[0]

    def value_0(self, items):
        return items[0]

    value_1 = value_0
    loopvar_0 = value_0
    collection_0 = value_0
    fieldref_0 = value_0

    def entity_0(self, items):
        return str(items[0])

    entity_1 = entity_2 = entity_3 = entity_4 = entity_5 = entity_0

    # Clauses: each one becomes a dictionary whose "kind" says what it is.
    def linkword_0(self, items):
        return None

    linkword_1 = linkword_0

    def relationclause_0(self, items):
        return {"kind": "relation", "predicate": items[1], "value": items[2]}

    def placeclause_0(self, items):
        return {"kind": "place", "value": items[0]}

    def duringclause_0(self, items):
        return {"kind": "during", "start": items[0], "end": items[1]}

    def whereclause_0(self, items):
        return {"kind": "where", "field": items[0], "operator": items[1], "value": items[2]}

    # The recursive rules become Python lists: the list so far plus one more element.
    def clauses_0(self, items):
        return items[0] + [items[1]]

    clauses_1 = clauses_0
    clauses_2 = clauses_0
    clauses_3 = clauses_0

    def clauses_4(self, items):
        return []

    def stmts_0(self, items):
        return items[0] + [items[1]]

    def stmts_1(self, items):
        return []

    def returnfields_0(self, items):
        return items[0] + [items[1]]

    def returnfields_1(self, items):
        return [items[0]]

    def returnclause_0(self, items):
        return items[0]

    # Queries. Without RETURN, the default field is name.
    def query_0(self, items):
        return {"kind": "query", "entity": items[0], "clauses": items[1], "return": ["name"]}

    def query_1(self, items):
        return {"kind": "query", "entity": items[0], "clauses": items[1], "return": items[2]}

    # Statements.
    def statement_0(self, items):
        return items[0]

    statement_1 = statement_2 = statement_3 = statement_0

    def letstmt_0(self, items):
        return {"kind": "let", "name": items[0], "query": items[1]}

    def querystmt_0(self, items):
        return {"kind": "query_stmt", "query": items[0]}

    def condition_0(self, items):
        return {"kind": "exists", "name": items[0]}

    def condition_1(self, items):
        return {"kind": "count_compare", "name": items[0], "operator": items[1], "value": items[2]}

    def ifstmt_0(self, items):
        return {"kind": "if", "condition": items[0], "then": items[1], "else": []}

    def ifstmt_1(self, items):
        return {"kind": "if", "condition": items[0], "then": items[1], "else": items[2]}

    def foreachstmt_0(self, items):
        return {"kind": "foreach", "item": items[0], "collection": items[1], "body": items[2]}

    # Top level: a RESEARCH block, or a plain sequence of statements.
    def researchblock_0(self, items):
        return {"kind": "research", "title": items[0], "statements": items[1]}

    def start_0(self, items):
        return items[0]

    def start_1(self, items):
        return {"kind": "program", "statements": items[0]}


def parse_histoql(source: str) -> dict[str, Any]:
    """Check a program against the attribute grammar and build its AST."""
    try:
        tree = parser.process(source)
    except UnexpectedInput as error:
        raise HistoQLSyntaxError(str(error)) from error
    except ContextConditionError as error:
        raise HistoQLSemanticError(str(error)) from error
    return ToAST().transform(tree)

### 4.3 SQL generation

Generated SQL is parameterised: values travel separately from the SQL text, and field names and operators come from closed lists enforced by the grammar. This protects the query against SQL injection.

In [6]:
# Demonstration schema. This adapter is the only part that knows it:
# entities(id, kind, name, role, occupation, birth_year, death_year, date, type, author, title, start_year, end_year, place_id)
# relationships(id, subject_id, predicate, object_id)
def compile_query_to_sql(query: dict[str, Any], dialect: str = "sqlite") -> tuple[str, list[Any]]:
    """Compile one query of a validated AST into SQL text and a separate list of parameter values."""
    mark = "?" if dialect == "sqlite" else "%s"  # placeholder style of each database driver

    select = ", ".join(f"e.{field}" for field in query["return"])
    joins, predicates, parameters = [], [f"e.kind = {mark}"], [query["entity"]]

    # The clause number keeps table aliases unique when a query repeats a clause.
    for index, clause in enumerate(query["clauses"], start=1):
        if clause["kind"] == "relation":
            # From the entity to its links, and from each link to the related entity.
            joins += [
                f"JOIN relationships r{index} ON r{index}.subject_id = e.id",
                f"JOIN entities o{index} ON o{index}.id = r{index}.object_id",
            ]
            predicates += [f"r{index}.predicate = {mark}", f"o{index}.name = {mark}"]
            parameters += [clause["predicate"], clause["value"]]
        elif clause["kind"] == "place":
            joins.append(f"JOIN entities p{index} ON p{index}.id = e.place_id")
            predicates.append(f"p{index}.name = {mark}")
            parameters.append(clause["value"])
        elif clause["kind"] == "during":
            # Two periods overlap if each one starts before the other ends. A missing end means a single year.
            predicates.append(f"e.start_year <= {mark} AND COALESCE(e.end_year, e.start_year) >= {mark}")
            parameters += [clause["end"], clause["start"]]
        elif clause["kind"] == "where":
            # The grammar already restricted the field and operator to closed lists; the value stays a parameter.
            predicates.append(f"e.{clause['field']} {clause['operator']} {mark}")
            parameters.append(clause["value"])

    join_sql = (" " + " ".join(joins)) if joins else ""
    sql = f"SELECT {select} FROM entities e{join_sql} WHERE {' AND '.join(predicates)};"
    return sql, parameters

In [7]:
# From a program to SQL: the first query of the example program.
example = '''
RESEARCH "Portuguese Liberalism" {
  LET politicians = FIND people
    WHO participated_in "Liberal Revolution"
    IN "Porto"
    DURING 1820..1834
    RETURN name, role;
  IF EXISTS politicians {
    FIND sources THAT mentions "Liberal Revolution" RETURN title, date;
  }
}
'''

ast = parse_histoql(example)  # rejects the program if the grammar does
first_query = ast["statements"][0]["query"]

# The same query for SQLite and for PostgreSQL: only the placeholder changes.
for dialect in ("sqlite", "postgresql"):
    sql, parameters = compile_query_to_sql(first_query, dialect=dialect)
    print(f"\n{dialect}: {sql}")
print("\nparameters:", parameters)


sqlite: SELECT e.name, e.role FROM entities e JOIN relationships r1 ON r1.subject_id = e.id JOIN entities o1 ON o1.id = r1.object_id JOIN entities p2 ON p2.id = e.place_id WHERE e.kind = ? AND r1.predicate = ? AND o1.name = ? AND p2.name = ? AND e.start_year <= ? AND COALESCE(e.end_year, e.start_year) >= ?;

postgresql: SELECT e.name, e.role FROM entities e JOIN relationships r1 ON r1.subject_id = e.id JOIN entities o1 ON o1.id = r1.object_id JOIN entities p2 ON p2.id = e.place_id WHERE e.kind = %s AND r1.predicate = %s AND o1.name = %s AND p2.name = %s AND e.start_year <= %s AND COALESCE(e.end_year, e.start_year) >= %s;

parameters: ['people', 'participated_in', 'Liberal Revolution', 'Porto', 1834, 1820]


### 4.4 Execution on demonstration data

The demonstration data is a small set we prepared for testing, not a complete archive. It has two datasets: the Liberal Revolution of 1820 and Luso-German trade during the Second World War. Each one is an `entities` file and a `relationships` file, loaded into an in-memory SQLite database when a program runs.

`FOR EACH` gives the loop variable the current element on each repetition. Because filters only accept literal values, the variable can be tested with `IF` but not yet used inside a filter.

In [8]:
DATASETS = {
    "liberalismo": ("entities.csv", "relationships.csv"),
    "ww2": ("trade_ww2_entities.csv", "trade_ww2_relationships.csv"),
}


def find_data_file(filename: str) -> str:
    """Look for a CSV file in the data folder next to the notebook, or in the parent folder."""
    for folder in ("data", ".", os.path.join("..", "data")):
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{filename} not found: place the CSV files in a 'data' folder next to this notebook")


def to_int(value: str | None) -> int | None:
    """An empty CSV cell becomes NULL in the database."""
    return int(value) if value else None


def create_archive_database(entities_csv: str, relationships_csv: str) -> sqlite3.Connection:
    """Create an in-memory SQLite database from one entities file and one relationships file."""
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE entities (
            id INTEGER PRIMARY KEY, kind TEXT NOT NULL, name TEXT, role TEXT, occupation TEXT,
            birth_year INTEGER, death_year INTEGER, date INTEGER, type TEXT, author TEXT, title TEXT,
            start_year INTEGER, end_year INTEGER, place_id INTEGER
        )""")
    cursor.execute("""
        CREATE TABLE relationships (
            id INTEGER PRIMARY KEY AUTOINCREMENT, subject_id INTEGER NOT NULL,
            predicate TEXT NOT NULL, object_id INTEGER NOT NULL
        )""")

    with open(find_data_file(entities_csv), encoding="utf-8") as file:
        for row in csv.DictReader(file):
            cursor.execute("INSERT INTO entities VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)", [
                int(row["id"]), row["kind"], row.get("name") or None, row.get("role") or None,
                row.get("occupation") or None, to_int(row.get("birth_year")), to_int(row.get("death_year")),
                to_int(row.get("date")), row.get("type") or None, row.get("author") or None,
                row.get("title") or None, to_int(row.get("start_year")), to_int(row.get("end_year")),
                to_int(row.get("place_id")),
            ])

    with open(find_data_file(relationships_csv), encoding="utf-8") as file:
        for row in csv.DictReader(file):
            cursor.execute("INSERT INTO relationships VALUES (?,?,?,?)", [
                int(row["id"]), int(row["subject_id"]), row["predicate"], int(row["object_id"]),
            ])

    conn.commit()
    return conn


def print_table(rows: list[dict[str, Any]], headers: list[str]) -> None:
    """Print results as an aligned table."""
    if not rows:
        print("  (no results)")
        return
    widths = {h: max(len(h), max(len(str(r.get(h, ""))) for r in rows)) for h in headers}
    print("  " + " | ".join(h.ljust(widths[h]) for h in headers))
    print("  " + "-+-".join("-" * widths[h] for h in headers))
    for row in rows:
        print("  " + " | ".join(str(row.get(h, "")).ljust(widths[h]) for h in headers))


def condition_holds(condition: dict[str, Any], stored: list[dict[str, Any]]) -> bool:
    """Evaluate EXISTS or COUNT over a stored result."""
    if condition["kind"] == "exists":
        return len(stored) > 0
    count, value = len(stored), condition["value"]
    return {
        "=": count == value, "!=": count != value,
        "<": count < value, "<=": count <= value,
        ">": count > value, ">=": count >= value,
    }[condition["operator"]]


def execute_program(source: str, conn: sqlite3.Connection, verbose: bool = True) -> dict[str, list[dict[str, Any]]]:
    """Run a whole program. The returned dictionary holds the stored results (the program state)."""
    ast = parse_histoql(source)
    if verbose:
        print("Syntax and semantics: OK")

    cursor = conn.cursor()
    env: dict[str, list[dict[str, Any]]] = {}

    def run_query(query: dict[str, Any]) -> list[dict[str, Any]]:
        sql, parameters = compile_query_to_sql(query, dialect="sqlite")
        cursor.execute(sql, parameters)
        return [dict(zip(query["return"], row)) for row in cursor.fetchall()]

    def run_statements(statements: list[dict[str, Any]]) -> None:
        for statement in statements:
            if statement["kind"] == "let":
                env[statement["name"]] = run_query(statement["query"])
                if verbose:
                    print(f"\n[LET {statement['name']}] Stored {len(env[statement['name']])} results:")
                    print_table(env[statement["name"]], statement["query"]["return"])
            elif statement["kind"] == "query_stmt":
                rows = run_query(statement["query"])
                if verbose:
                    print("\n[QUERY] Results:")
                    print_table(rows, statement["query"]["return"])
            elif statement["kind"] == "if":
                name = statement["condition"]["name"]
                holds = condition_holds(statement["condition"], env.get(name, []))
                if verbose:
                    print(f"\n[IF {name}] Condition '{statement['condition']['kind']}' is {holds}")
                run_statements(statement["then"] if holds else statement["else"])
            elif statement["kind"] == "foreach":
                collection = env.get(statement["collection"], [])
                if verbose:
                    print(f"\n[FOR EACH] {len(collection)} elements in '{statement['collection']}'")
                for row in collection:
                    env[statement["item"]] = [row]
                    run_statements(statement["body"])

    if ast["kind"] == "research" and verbose:
        print(f"Research project: '{ast['title']}'")
    run_statements(ast.get("statements", []))
    return env

### 4.5 Running a program

`histoql()` is the entry point of the implementation. It runs a program given as text or as a file, on the dataset chosen, on a pair of CSV files of our own, or on the dataset detected from the program. With `sql_only=True` it checks the program and shows the SQL without executing it.

In [9]:
# Words that point to the wartime trade dataset. Any other program uses the Liberal Revolution dataset.
WW2_KEYWORDS = [
    "193", "194", "volframio", "alemanha", "reich", "carvao", "ouro", "conservas", "sardinha",
    "mauser", "panasqueira", "borralha", "mercadoria", "comercio", "clearing", "wolfram", "germany",
]


def detect_dataset(source: str) -> str:
    """Choose a dataset from the content of the program. This is a heuristic: dataset= overrides it."""
    lowered = source.lower()
    return "ww2" if any(keyword in lowered for keyword in WW2_KEYWORDS) else "liberalismo"


def collect_queries(statements: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Find every query of a program, including those inside blocks."""
    queries = []
    for statement in statements:
        if "query" in statement:
            queries.append(statement["query"])
        for block in ("then", "else", "body"):
            queries += collect_queries(statement.get(block, []))
    return queries


def histoql(source: str | None = None, file: str | None = None, dataset: str = "auto",
            entities: str | None = None, relationships: str | None = None, sql_only: bool = False):
    """Run a HistoQL program, given as text or as a file."""
    if file is not None:
        with open(file, encoding="utf-8") as handle:
            source = handle.read()

    if sql_only:
        ast = parse_histoql(source)
        print("Syntax and semantics: OK")
        for query in collect_queries(ast.get("statements", [])):
            sql, parameters = compile_query_to_sql(query)
            print(f"\nSQL: {sql}\nParameters: {parameters}")
        return None

    if entities and relationships:
        entities_csv, relationships_csv = entities, relationships
    else:
        chosen = detect_dataset(source) if dataset == "auto" else dataset
        entities_csv, relationships_csv = DATASETS[chosen]
    return execute_program(source, create_archive_database(entities_csv, relationships_csv))

### 4.6 Demonstration

The next cells run programs from start to finish. First, the example program of section 1:

In [10]:
histoql(example);

Syntax and semantics: OK
Research project: 'Portuguese Liberalism'

[LET politicians] Stored 5 results:
  name                                      | role                              
  ------------------------------------------+-----------------------------------
  Manuel Fernandes Tomás                    | Juiz e Líder Revolucionário       
  José da Silva Carvalho                    | Ministro e Membro da Junta        
  Bernardo Sepúlveda                        | Coronel e Comandante Militar      
  Sebastião Drago Valente de Brito Cabreira | Coronel de Artilharia             
  Frei Francisco de São Luís                | Secretário dos Negócios Interiores

[IF politicians] Condition 'exists' is True

[QUERY] Results:
  title                                                 | date
  ------------------------------------------------------+-----
  Manifesto da Junta Provisional aos Portugueses        | 1820
  Gazeta de Lisboa - Edição Extraordinária              | 1820
  Constituição

The second dataset, on Luso-German trade in 1940 and 1941:

In [11]:
WW2_PROGRAM = """
RESEARCH "Luso-German Wartime Trade 1940-1941" {
  LET mercadorias = FIND events
    DURING 1940..1941
    RETURN name, date, type;

  IF EXISTS mercadorias {
    FIND sources THAT mentions "Exportacao Macica de Volframio para a Alemanha" RETURN title, author, date;
  }
}
"""
histoql(WW2_PROGRAM);

Syntax and semantics: OK
Research project: 'Luso-German Wartime Trade 1940-1941'

[LET mercadorias] Stored 6 results:
  name                                                  | date | type                 
  ------------------------------------------------------+------+----------------------
  Exportacao de Conservas de Sardinha para a Alemanha   | 1940 | mercadoria_exportacao
  Importacao de Carvao da Bacia do Ruhr                 | 1940 | mercadoria_importacao
  Importacao de Ferro e Aco Laminado Alemao             | 1940 | mercadoria_importacao
  Exportacao Macica de Volframio para a Alemanha        | 1941 | mercadoria_exportacao
  Importacao de Armamento Mauser e Municoes             | 1941 | mercadoria_importacao
  Liquidacao de Saldos Comerciais em Ouro do Reichsbank | 1941 | operacao_cambial     

[IF mercadorias] Condition 'exists' is True

[QUERY] Results:
  title                                                               | author                            | date
  --------

A filter on a field, with no block around the query:

In [12]:
histoql('FIND events WHERE type = "mercadoria_exportacao" RETURN name, date;', dataset="ww2");

Syntax and semantics: OK

[QUERY] Results:
  name                                                | date
  ----------------------------------------------------+-----
  Exportacao de Conservas de Sardinha para a Alemanha | 1940
  Exportacao Macica de Volframio para a Alemanha      | 1941


`ELSE`: only one branch runs. In the second program no person matches, so the condition is false.

In [13]:
histoql('LET a = FIND people; IF EXISTS a { FIND places WHERE name = "Lisboa"; } ELSE { FIND places WHERE name = "Porto"; }', dataset="liberalismo");
print()
histoql('LET a = FIND people WHERE name = "nobody"; IF EXISTS a { FIND places WHERE name = "Lisboa"; } ELSE { FIND places WHERE name = "Porto"; }', dataset="liberalismo");

Syntax and semantics: OK

[LET a] Stored 8 results:
  name                                     
  -----------------------------------------
  Manuel Fernandes Tomás                   
  José da Silva Carvalho                   
  Bernardo Sepúlveda                       
  Sebastião Drago Valente de Brito Cabreira
  Frei Francisco de São Luís               
  D. Pedro IV                              
  D. Miguel I                              
  Manuel Borges Carneiro                   

[IF a] Condition 'exists' is True

[QUERY] Results:
  name  
  ------
  Lisboa

Syntax and semantics: OK

[LET a] Stored 0 results:
  (no results)

[IF a] Condition 'exists' is False

[QUERY] Results:
  name 
  -----
  Porto


`FOR EACH` with a nested `IF` on the loop variable:

In [14]:
histoql('LET founders = FIND people IN "Porto" DURING 1820..1820 RETURN name; FOR EACH founder IN founders { IF EXISTS founder { FIND places WHERE name = "Porto"; } }', dataset="liberalismo");

Syntax and semantics: OK

[LET founders] Stored 5 results:
  name                                     
  -----------------------------------------
  Manuel Fernandes Tomás                   
  José da Silva Carvalho                   
  Bernardo Sepúlveda                       
  Sebastião Drago Valente de Brito Cabreira
  Frei Francisco de São Luís               

[FOR EACH] 5 elements in 'founders'

[IF founder] Condition 'exists' is True

[QUERY] Results:
  name 
  -----
  Porto

[IF founder] Condition 'exists' is True

[QUERY] Results:
  name 
  -----
  Porto

[IF founder] Condition 'exists' is True

[QUERY] Results:
  name 
  -----
  Porto

[IF founder] Condition 'exists' is True

[QUERY] Results:
  name 
  -----
  Porto

[IF founder] Condition 'exists' is True

[QUERY] Results:
  name 
  -----
  Porto


SQL only, without executing the program:

In [15]:
histoql(example, sql_only=True);

Syntax and semantics: OK

SQL: SELECT e.name, e.role FROM entities e JOIN relationships r1 ON r1.subject_id = e.id JOIN entities o1 ON o1.id = r1.object_id JOIN entities p2 ON p2.id = e.place_id WHERE e.kind = ? AND r1.predicate = ? AND o1.name = ? AND p2.name = ? AND e.start_year <= ? AND COALESCE(e.end_year, e.start_year) >= ?;
Parameters: ['people', 'participated_in', 'Liberal Revolution', 'Porto', 1834, 1820]

SQL: SELECT e.title, e.date FROM entities e JOIN relationships r1 ON r1.subject_id = e.id JOIN entities o1 ON o1.id = r1.object_id WHERE e.kind = ? AND r1.predicate = ? AND o1.name = ?;
Parameters: ['sources', 'mentions', 'Liberal Revolution']


The same errors through `histoql()`: a program that the grammar rejects never reaches SQL.

In [16]:
for program in ('FIND people RETURN name', 'FIND people DURING 1834..1820 RETURN name;', 'IF EXISTS fantasmas { FIND people RETURN name; }'):
    try:
        histoql(program, dataset="liberalismo")
    except (HistoQLSyntaxError, HistoQLSemanticError) as error:
        print(f"{type(error).__name__}: {str(error).splitlines()[0]}\n  in: {program}")

HistoQLSyntaxError: Unexpected token Token('$END', '') at line 1, column 20.
  in: FIND people RETURN name
HistoQLSemanticError: Invalid period
  in: FIND people DURING 1834..1820 RETURN name;
HistoQLSemanticError: Undeclared research result
  in: IF EXISTS fantasmas { FIND people RETURN name; }


## 5. Language and grammar quality

We evaluate the language and its grammar using the internal factors and grammar metrics of notebook NB07.

### Language quality

| Factor | Evaluation |
|---|---|
| Expressiveness | Strong for the queries we target, which read close to the research question. There is no `OR`, no negation and no comparison between fields. |
| Documentation | Keywords are readable, `RESEARCH` gives a program a title, and `#` allows comments. |
| Uniqueness | `WHO` and `THAT` are synonyms: this improves reading but gives two ways of saying the same thing. We accepted this trade-off. |
| Consistency | Letter case and clause structure are uniform. The exception is the default `RETURN`, which is `name`, a field `sources` does not have; a context condition reports this. |
| Extensibility | Fields and relations are lists rather than productions, and each list appears once in the grammar. |
| Scalability | There are no functions or modules, which is acceptable at this scale but limits large programs. |
| Reliability | Inverted periods and invalid fields, relations and names are detected before execution. Value types in `WHERE` are not checked. |
| Modularity | SQL generation is isolated in an adapter. The language itself has no modules. |

### Grammar metrics

We measured the grammar through Lark.

| Metric | Value |
|---|---|
| Terminals | 33 |
| Non-terminals | 24 |
| Productions | 44 |
| Unit productions | 7 |
| Average right-hand side length | 2.18 |
| Longest right-hand side | 9 |
| LALR states | 84 |
| LALR conflicts | 0 |
| Attributes | 21 (12 inherited, 9 synthesized) |
| Evaluation rules | 50 |
| Context conditions | 9 |

The grammar is pure BNF, and direct recursion is entirely on the left, in `stmts`, `clauses` and `returnfields`. Pure BNF is longer to read than EBNF, but Lark does not need to generate auxiliary rules for it, and every list step is a node that can carry attributes. The longest production is the `IF ... ELSE`. Having no LALR conflicts means the grammar is unambiguous. As NB07 notes, a metric is evidence and does not by itself make a grammar better or worse.

### Known limitations

| Limitation | Effect | Direction |
|---|---|---|
| `FOR EACH` variable | It cannot be used inside a filter | Let `value` accept an `ID` |
| Types in `WHERE` | `WHERE name >= 5` is accepted | Synthesize a type for `value` and `fieldref`, and compare them in a context condition |
| Size of the vocabulary lists | The lists of fields and relations are written inside context conditions, which makes those rules long | Keep them in a table that the grammar reads |

### Part II

The language will be reused in Part II, where programs will be represented, analysed, visualized and transformed. The AST and the relations between entities are our starting point.